# Long-Shot — a quantitative teardown 🔬
### The skewness long-short with a Lo t-stat · persistence · the cost sweep · the breadth limit

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Skewness effect is real?: Supported](https://img.shields.io/badge/Skewness_effect_is_real%3F-Supported-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). The commodity skewness effect leans the right way but is under-powered by a small basket.

> ⚠️ **Not investment advice.** 14 commodity ETFs, daily, 2009–2026 (Yahoo). Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (long_shot/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from long_shot import data, strategy as st
daily = data.fetch_panel()                     # cache-first
sig = st.skewness_signal(daily)
h = st.cross_section_hedge(daily, sig, long_high=False)   # long low-skew


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Weak** | +5.2%/yr, Sharpe 0.27, Lo t 1.1 (14 names) |
| Tradability | **Fragile** | net Sharpe 0.19 at 10 bp |
| Effect real? | **Supported** | right sign, stable, literature |

> 💡 *In plain words:* the lottery story holds in commodities — faintly.

## 1 · The claim, steelmanned

- **H₁:** long-low/short-high-skew earns a positive premium.
- **H₂:** it's significant.
- **H₃:** it's stable / doesn't invert (unlike the equity version).

## 2 · So what? — what rides on each

If H₁/H₃ hold, the lottery bias is a tradable commodity premium. H₂ decides whether 14 ETFs can prove it.

## 3 · How we'd know — the protocol

Trailing-year skewness → long-low/short-high → Lo t-stat → decade split → cost sweep.

## 4 · The teardown

### 4.1 The long-short

In [2]:
import pandas as pd; display(pd.Series(st.stats(h)).round(3))

mean_ann      0.052
sharpe        0.269
tstat         1.123
hit_rate      0.529
n           210.000
dtype: float64

> 💡 *In plain words:* +5.2%/yr but t 1.1 — **H₁ holds, H₂ rejected** on this small basket.

### 4.2 Persistence (not inverted, not decayed)

In [3]:
for lab,sl in [('2009-2017',h.loc[:'2017']),('2018-on',h.loc['2018':])]:
    print(f'{lab}: Sharpe {st.stats(sl)["sharpe"]:+.2f}, mean {st.stats(sl)["mean_ann"]:+.2%}/yr')

2009-2017: Sharpe +0.28, mean +4.33%/yr
2018-on: Sharpe +0.27, mean +6.18%/yr


> 💡 *In plain words:* +0.28 then +0.27 — stable and right-signed. **H₃ holds** (contrast the inverted equity [53 Jackpot](../../53-jackpot/)).

### 4.3 Costs and the breadth limit

In [4]:
for c in (0,10,25): print(f'net Sharpe @ {c}bp: {st.stats(st.net_of_cost(h,c))["sharpe"]:+.2f}')
print('commodities in basket:', daily.shape[1])

net Sharpe @ 0bp: +0.27
net Sharpe @ 10bp: +0.19
net Sharpe @ 25bp: +0.08
commodities in basket: 14


> 💡 *In plain words:* costs pull it toward zero, and 14 ETFs (tercile baskets of ~5) starve the t-stat. The fix is breadth (a futures universe), not direction.

## 5 · The verdict

H₁/H₃ hold, H₂ rejected (breadth) → Signal `WEAK`, Tradability `FRAGILE`, the effect `SUPPORTED`.

## 6 · Could you trade it?

With a broad commodity-futures universe and cheap execution, plausibly — it's right-signed and persistent. On 14 ETFs with roll drag, real but thin.

## 7 · Going further

Forks: (a) a 20–30 commodity-futures universe (the literature's breadth); (b) combine skewness with term-structure/momentum (Fuertes et al.'s multi-signal book); (c) value-weight by liquidity. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).